# 棱镜大模型 QLoRA 微调
自动下载训练数据 + 微调Qwen2.5-7B-Instruct + 保存模型

**运行前请确认：运行时 → 更改运行时类型 → T4 GPU**

In [ ]:
!pip install -q unsloth
!pip install -q --no-deps trl peft accelerate bitsandbytes
!pip install -q transformers datasets

In [ ]:
import os
DATA_DIR = '/content/prism_training'
os.makedirs(DATA_DIR, exist_ok=True)
!wget -q https://raw.githubusercontent.com/prism-invest-hqhub/prism-capital/main/data/training_data_deduped.jsonl -O {DATA_DIR}/training_data.jsonl

with open(f'{DATA_DIR}/training_data.jsonl') as f:
    lines = [l.strip() for l in f if l.strip()]
print(f'训练数据: {len(lines)} 条')

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='Qwen/Qwen2.5-7B-Instruct',
    max_seq_length=2048,
    load_in_4bit=True,
    dtype=None,
)
print('模型加载完成')

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=32, lora_alpha=64, lora_dropout=0.05,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)
print('LoRA配置完成')

In [ ]:
from datasets import Dataset
import json

def format_to_chatml(example):
    messages = []
    if 'instruction' in example and 'output' in example:
        if example.get('input'):
            messages = [{'role':'user','content':f"{example['instruction']}\n{example['input']}"},
                       {'role':'assistant','content':example['output']}]
        else:
            messages = [{'role':'user','content':example['instruction']},
                       {'role':'assistant','content':example['output']}]
    elif 'conversations' in example:
        for conv in example['conversations']:
            role = 'user' if conv['from'] == 'human' else 'assistant'
            messages.append({'role':role,'content':conv['value']})
    elif 'messages' in example:
        messages = example['messages']
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {'text': text}

raw_data = []
with open(f'{DATA_DIR}/training_data.jsonl') as f:
    for line in f:
        if line.strip():
            raw_data.append(json.loads(line.strip()))

dataset = Dataset.from_list(raw_data)
dataset = dataset.map(format_to_chatml, remove_columns=dataset.column_names)
dataset = dataset.filter(lambda x: len(x['text']) > 50)
print(f'有效训练样本: {len(dataset)}')

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer,
    train_dataset=dataset, dataset_text_field='text',
    max_seq_length=2048,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=50,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=25,
        save_steps=200,
        output_dir=f'{DATA_DIR}/checkpoints',
        optim='adamw_8bit',
        seed=42,
        report_to='none',
    ),
)
trainer.train()
print('训练完成！')

In [ ]:
OUTPUT_DIR = f'{DATA_DIR}/prism-7b-qlora-v1'
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'LoRA适配器已保存: {OUTPUT_DIR}')

MERGED_DIR = f'{DATA_DIR}/prism-7b-merged-v1'
try:
    model.save_pretrained_merged(MERGED_DIR, tokenizer, save_method='merged_16bit')
    print(f'合并模型已保存: {MERGED_DIR}')
except Exception as e:
    print(f'合并失败（显存不足）: {e}\nLoRA适配器已单独保存')